In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

# Load silver data
df = spark.read.table("subscription_pipeline.silver.opportunity")

# Window for customer history
window_spec = Window.partitionBy("customer_id").orderBy("start_date")

fact = df.withColumn(
    "prev_revenue",
    lag("revenue_amount").over(window_spec)
).withColumn(
    "revenue_change",
    col("revenue_amount") - col("prev_revenue")
).withColumn(
    "is_new_customer",
    when(row_number().over(window_spec) == 1, True).otherwise(False)
).withColumn(
    "is_upgrade",
    when(col("revenue_change") > 0, True).otherwise(False)
).withColumn(
    "is_downgrade",
    when(col("revenue_change") < 0, True).otherwise(False)
).withColumn(
    "subscription_status",
    col("close_status")
).withColumn(
    "year",
    year("start_date")
).withColumn(
    "month",
    month("start_date")
)

# Final select
fact_final = fact.select(
    "opportunity_id",
    "customer_id",
    "product_id",
    "start_date",
    "end_date",
    col("revenue_amount").alias("revenue_gbp"),
    "prev_revenue",
    "revenue_change",
    "is_new_customer",
    "is_upgrade",
    "is_downgrade",
    "subscription_status",
    "year",
    "month"
)

# Save table
fact_final.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("subscription_pipeline.gold.fact_subscription")